# 300 m Walkability: Isochrone Computation (DuckDB version)

= For each residential address in a city, determine whether it lies within **300 m walking distance** of a publicly accessible park or garden = 

Park/garden = define as any size public greenery in the polygon of given city. Public restrain is given by 1.OSM labels, 2.walking path x park border crossing

**Pipeline:**
1. Load all data from DuckDB (boundary, civici, parks): *raw schema in duckdb*
2. Download OSM pedestrian network (OSMnx)
3. Detect park entrance points (3-options: OSM tags, boundary ∩ street, centroid snap(osm only))
4. Compute 300 m walking isochrones from each entrance (GraphHopper)
5. Spatial join: which civic numbers fall inside an isochrone?
6. Point-to-point routing: confirm and measure the walk distance
7. Write results back to DuckDB + export 

> **GraphHopper must be running** before cells 11–13.  
> ```bash
> rm -rf graph-cache && java -jar graphhopper-web-11.0.jar server config.yml
> ```

In [ ]:
!pip install python-polylabel

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import duckdb
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import MultiPoint, Point, shape
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from shapely.validation import make_valid
import os
import sys
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
from polylabel import polylabel
import pyproj
from shapely.ops import transform as shp_transform



In [ ]:
from config.config import (ROOT_PATH, EUCLID_CUTOFF, connect_db, gdf_to_duckdb, load_table, gpkg_to_duckdb,get_city_cfg, get_output_paths,
        DB_PATH, GH_PORT, CRS_WGS, CRS_METRIC, WALKING_DISTANCE,
)

os.chdir(ROOT_PATH)

if ROOT_PATH not in sys.path:
    sys.path.insert(0, ROOT_PATH)

In [4]:
con = connect_db()

In [ ]:
CITY    = "torino"          
cfg     = get_city_cfg(CITY)
src     = cfg["data_sources"]
paths   = get_output_paths(CITY)

GH_URL  = f"http://localhost:{GH_PORT}/route"
ISO_URL = f"http://localhost:{GH_PORT}/isochrone"

print(f"City: {CITY}")
print(f"GH URL: {GH_URL}")
print(f"DB path: {DB_PATH}")

OSM_SNAP_TOL = 2.0   # OSM node must be within 2 m of park boundary
DEDUP_DIST = 5.0   # cluster intersection points in etrance detection closer than 5 m 

City    : torino
GH URL  : http://localhost:8989/route
DB path : /Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index/config/3_30_300.duckdb


## 1 Load layers from DuckDB

In [ ]:
parks = load_table(con, "preprocessing", f"{CITY}_parks_greenery")
parks["geometry"] = parks["geometry"].apply(make_valid)
print(f"Parks  : {len(parks):,}  CRS={parks.crs}")

result = con.execute(f"""
    SELECT
        source,
        COUNT(*)        AS n_parks,
        SUM(area_ha)    AS total_ha
    FROM preprocessing.{CITY}_parks_greenery
    GROUP BY source
""").fetchall()

for row in result:
    print(f"{row[0]:<40} {row[1]:>6,} parks   {row[2]:>8.0f} ha")
    


torino_parks_osm                            864 parks       1235 ha
torino_parks_dsbn_pe_unis                     1 parks         20 ha
Parks  : 865  CRS=EPSG:32632


In [7]:
boundary = load_table(con, "raw", f"{CITY}_boundary")
boundary_wgs84 = boundary.to_crs(CRS_WGS).geometry.union_all()
print(f"Boundary area  : {boundary_wgs84.area:.6f} deg²")
print(f"Boundary bounds: {boundary_wgs84.bounds}")

Boundary area  : 0.014864 deg²
Boundary bounds: (7.577834800000002, 45.0067924, 7.773338800000001, 45.140217499999984)


In [8]:
# "buildings_sn" key  => raw.milano_buildings_sn
civici = load_table(con, "raw", f"{CITY}_buildings_sn")

# Use pre-computed building_id if it exists, else derive it - this is important as it differe between data for different cities
if "building_id" in civici.columns:
    civici["building_id"] = civici["building_id"].astype(str)
else:
    civici["building_id"] = (
        civici["CODICE_VIA"].astype(str).str.strip() + "_"
        + civici[cfg["building_id_col"]].astype(str).str.strip() + "_"
        + civici.get("LETTERA", pd.Series("", index=civici.index)).fillna("").astype(str)
    )

civici = civici.reset_index(drop=True)
assert civici["building_id"].nunique() == len(civici), \
    f"building_id not unique! {civici['building_id'].nunique()} unique / {len(civici)} rows"


## 2  Pedestrian network (OSMnx)

In [9]:
print("Downloading walk network: ")
G = ox.graph_from_polygon(boundary_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)
_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"Pedestrian edges from OSM: {len(edges):,}  CRS={edges.crs}")

Pedestrian edges from OSM: 91,226  CRS=EPSG:32632


## 3  OSM entrance nodes

In [10]:
print("Get OSM entrance: ")
osm_entrances_raw = ox.features_from_polygon(
    boundary_wgs84,
    tags={"entrance": ["yes", "main", "secondary", "service",
                       "staircase", "emergency", "exit"]},
).to_crs(CRS_METRIC)

osm_entrance_pts = osm_entrances_raw[
    osm_entrances_raw.geometry.geom_type == "Point"
].copy()
print(f"Number of OSM entrance: {len(osm_entrance_pts):,}")

Get OSM entrance: 
Number of OSM entrance: 1,349


## 4  Entrance detection

In [11]:
def extract_points(geom) -> list:
    """Extract all Point objects from any Shapely geometry type."""
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t == "Point":
        return [geom]
    if t == "MultiPoint":
        return list(geom.geoms)
    if t == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []


#Merge points within 5 m into one centroid = has been tested
#Give very simualr results, accept this makes the computation significantly faster for big cities.
#Can be skipped - used only in tier 2

def cluster_points(pts: list, dist: float) -> list:
    
    if not pts:
        return []
    remaining = list(pts)
    clusters  = []
    while remaining:
        seed  = remaining.pop(0)
        group = [seed]
        far   = []
        for p in remaining:
            (group if seed.distance(p) <= dist else far).append(p)
        remaining = far
        clusters.append(MultiPoint(group).centroid)
    return clusters


## 5 Detect park entrances 

| Tier | Method |
|------|--------|
| 1 | Explicit `entrance=*` OSM node |
| 2 | Intersection of park boundary with a walking path|
| 3 | OSM only: Sample boundary at regular intervals, find closest edge point|

*More honest than centroid: places the entrance ON the boundary
    near actual infrastructure 



In [ ]:
"""
def get_interior_point(geom):
    try:
        if geom.geom_type == "MultiPolygon":
            geom = max(geom.geoms, key=lambda g: g.area)
        coords = [list(geom.exterior.coords)] + [list(ring.coords) for ring in geom.interiors]
        result = polylabel(coords, 1.0)  # positional only, no keyword argument
        return Point(result[0], result[1])
    except Exception:
        return None
"""

In [12]:
def get_boundary_entrance(geom, edge_tree, edges_geom, search_radius=50):
    """
    Sample boundary at regular intervals, find closest edge point.
    More honest than centroid — places the entrance ON the boundary
    near actual infrastructure.
    """
    boundary = geom.boundary
    # Sample every 10m along boundary
    length = boundary.length
    n_samples = max(8, int(length / 10))
    samples = [boundary.interpolate(i / n_samples, normalized=True) 
               for i in range(n_samples)]
    
    best_pt, best_dist = None, float("inf")
    for pt in samples:
        hits = edge_tree.query(pt.buffer(search_radius), predicate="intersects")
        for idx in hits:
            edge_pt, _ = nearest_points(edges_geom[idx], pt)
            d = pt.distance(edge_pt)
            if d < best_dist:
                best_dist = d
                best_pt = pt  # snap to boundary, not to edge
    return best_pt

In [35]:
edges_geom = edges.geometry.values
edge_mids  = np.array([
    [g.interpolate(0.5, normalized=True).x,
     g.interpolate(0.5, normalized=True).y]
    for g in edges_geom
])

osm_entrance_tree = STRtree(osm_entrance_pts.geometry.values)
edge_tree = STRtree(edges_geom)

rows = []
step_counts = {1: 0, 2: 0, 3: 0, "3_extended": 0}

for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    geom      = park.geometry
    park_bdry = geom.boundary      # per-park LineString 
    park_id   = park["park_id"]
    pts       = []

    # Tier 1: explicit OSM entrance=* node within OSM_SNAP_TOL of boundary
    osm_hits  = osm_entrance_tree.query(geom.buffer(OSM_SNAP_TOL),
                                        predicate="intersects")
    tier1_pts = [
        osm_entrance_pts.geometry.iloc[i]
        for i in osm_hits
        if park_bdry.distance(osm_entrance_pts.geometry.iloc[i]) <= OSM_SNAP_TOL
    ]
    if tier1_pts:
        pts = tier1_pts
        step_counts[1] += 1

    # Tier 2: boundary ∩ external edges + near-miss projection
    if not pts:
        tier2_raw = []
        for idx in edge_tree.query(park_bdry, predicate="intersects"):
            if not geom.contains(edges_geom[idx]):
                inter = park_bdry.intersection(edges_geom[idx])
                tier2_raw.extend(extract_points(inter))

        near_miss_idxs = [
            idx for idx in edge_tree.query(park_bdry.buffer(5), predicate="intersects")
            if not geom.contains(edges_geom[idx])
            and not park_bdry.intersects(edges_geom[idx])
        ]
        for idx in near_miss_idxs:
            proj_pt, _ = nearest_points(edges_geom[idx], park_bdry)
            tier2_raw.append(proj_pt)

        if tier2_raw:
            pts = cluster_points(tier2_raw, DEDUP_DIST)
            step_counts[2] += 1

    if not pts:
        if park["source"] == f"{CITY}_parks_osm" or (pd.notna(park.get("name")) and park.get("name")):
            snap = get_boundary_entrance(geom, edge_tree, edges_geom)
            #snap = get_interior_point(geom)

            if snap:
                pts.append(snap)
                if park["source"] == f"{CITY}_parks_osm":
                    step_counts[3] += 1
                else:
                    step_counts["3_extended"] += 1

    transformer = pyproj.Transformer.from_crs(CRS_METRIC, CRS_WGS, always_xy=True)

    for pt in pts:
        lon, lat = transformer.transform(pt.x, pt.y)
        rows.append({
            "park_id":      park_id,
            "entrance_lat": lat,
            "entrance_lon": lon,
            "geometry":     pt,
        })


entrances_df = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_METRIC)

missing = set(parks["park_id"]) - set(entrances_df["park_id"])
print(f"\n{len(entrances_df):,} entrance points across {entrances_df['park_id'].nunique()} parks")
print(f"OSM: {step_counts[1]} | boundary∩street: {step_counts[2]} | "
      f"centroid: {step_counts[3]} | extended: {step_counts['3_extended']}")
print(f"Parks with no entrance: {len(missing)}")


Park entrances: 100%|██████████| 865/865 [00:15<00:00, 56.47it/s]


3,089 entrance points across 848 parks
OSM: 17 | boundary∩street: 549 | centroid: 282 | extended: 0
Parks with no entrance: 17


In [ ]:
#check parks without entrance
if missing:
    print("  Missing IDs:", sorted(missing))

In [37]:
# Save entrances to DuckDB
gdf_to_duckdb(con, entrances_df, f"preprocessing.{CITY}_entrances")

preprocessing.torino_entrances: 3089 rows written


## 6  Civici inside a park

In [38]:
civici_in_park = con.execute(f"""
    SELECT c.building_id, p.park_id
    FROM raw.{CITY}_buildings_sn c
    JOIN preprocessing.{CITY}_parks_greenery p
      ON ST_Within(c.geometry, p.geometry)
""").df()

civici_in_park_ids    = set(civici_in_park["building_id"])
civici_in_park_lookup = civici_in_park.set_index("building_id")["park_id"].to_dict()
print(f"Civici inside a park polygon: {len(civici_in_park_ids):,}")

Civici inside a park polygon: 640


## 7  GraphHopper: isochrones

In [39]:
#walking isochrone from GraphHopper
#Returns = polygon feature dict

def get_isochrone(lat: float, lon: float, distance_m: int = 300):
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None

In [40]:
iso_rows = []
for _, row in tqdm(entrances_df.iterrows(), total=len(entrances_df), desc="Isochrones"):
    result = get_isochrone(row["entrance_lat"], row["entrance_lon"])
    if result:
        iso_rows.append({
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "geometry":     shape(result["geometry"]),
        })

# Isochrones from GH are WGS84 = reproject to metric for spatial join!
isochrones = gpd.GeoDataFrame(
    iso_rows,
    geometry="geometry",
    crs=CRS_WGS,
).to_crs(CRS_METRIC)

print(f"{len(isochrones):,} isochrones computed"
      f"({len(entrances_df) - len(isochrones):,} failed)")

#save intermediate results to
gdf_to_duckdb(con, isochrones, f"preprocessing.{CITY}_isochrones")


Isochrones: 100%|██████████| 3089/3089 [05:20<00:00,  9.63it/s]


3,089 isochrones computed(0 failed)
preprocessing.torino_isochrones: 3089 rows written


## 8 Spatial join: civici within isochrones

In [41]:
civici_gdf = civici[["building_id", "geometry"]].copy().reset_index(drop=True)

In [42]:
print(cfg["building_id_col"])
print(civici[cfg["building_id_col"]].nunique())
print(civici[cfg["building_id_col"]].value_counts().head(5))

NUMEROCOMPLETO


KeyError: 'NUMEROCOMPLETO'

In [43]:
joined = gpd.sjoin(
    civici_gdf,
    isochrones[["park_id", "entrance_lat", "entrance_lon", "geometry"]],
    how="left",
    predicate="within",
)

candidates = joined[joined["park_id"].notna()].copy().reset_index(drop=True)

print(f"Raw join rows : {len(joined):,}")
print(f"Matched (in iso) : {len(candidates):,}")
print(f"Unmatched : {joined['park_id'].isna().sum():,}")
print(f"Unique civici matched: {candidates['building_id'].nunique():,} "
      f"({candidates['building_id'].nunique()/len(civici)*100:.1f}%)")

Raw join rows : 622,393
Matched (in iso) : 584,569
Unmatched : 37,824
Unique civici matched: 81,000 (68.2%)


In [46]:
cands_wgs = gpd.GeoDataFrame(
    candidates[["building_id"]],
    geometry=candidates.geometry,
    crs=CRS_METRIC,
).to_crs(CRS_WGS) # WGS84 coords of each candidate civico (for GH routing)

candidates = candidates.copy()
candidates["orig_lat"] = cands_wgs.geometry.y
candidates["orig_lon"] = cands_wgs.geometry.x

#Euclidean pre-filter - 420 m - so it is lighter to process the routing later
entrance_pts_metric = gpd.GeoDataFrame(
    candidates[["entrance_lat", "entrance_lon"]],
    geometry=gpd.points_from_xy(candidates["entrance_lon"], candidates["entrance_lat"]),
    crs=CRS_WGS,
).to_crs(CRS_METRIC)

candidates["euclid_m"] = candidates.geometry.distance(entrance_pts_metric.geometry)

before = len(candidates)
candidates = (
    candidates[candidates["euclid_m"] <= EUCLID_CUTOFF]
    .copy()
    .reset_index(drop=True)
)
after = len(candidates)
print(f"Euclidean pre-filter: {before:,} = {after:,} "
      f"(dropped {before-after:,}, kept {after/before*100:.1f}%)")

# Per-civico cap: keep only the N closest entrances = lighter processing 
MAX_ENTRANCES_PER_CIVICO = 5
candidates = (
    candidates
    .sort_values("euclid_m")
    .groupby("building_id", sort=False)
    .head(MAX_ENTRANCES_PER_CIVICO)
    .reset_index(drop=True)
)

n_pairs = len(candidates)
n_cands = candidates["building_id"].nunique()
print(f"{n_cands:,} candidate civici ({n_cands/len(civici)*100:.1f}% of total)")
print(f"{n_pairs:,} (civico, entrance) pairs to route")
print(f"avg {n_pairs/max(n_cands,1):.1f} entrance candidates per civico")

Euclidean pre-filter: 584,569 = 584,568 (dropped 1, kept 100.0%)
81,000 candidate civici (68.2% of total)
306,436 (civico, entrance) pairs to route
avg 3.8 entrance candidates per civico


## 9 Point-to-point routing (GraphHopper)

In [44]:
#Walking distance in metres between two WGS84 points

def graphhopper_dist(orig_lat, orig_lon, dest_lat, dest_lon):
    params = {
        "point":        [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":      "foot",
        "calc_points":  "false",
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

In [ ]:
rows = []
for _, row in tqdm(candidates.iterrows(), total=n_pairs, desc="Routing"):
    dist = graphhopper_dist(
        row["orig_lat"], row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"],
    )
    if dist is not None:
        rows.append((row["building_id"], row["park_id"],
                     row["entrance_lat"], row["entrance_lon"], dist))

print(f"{len(rows):,} pairs routed  ({n_pairs - len(rows):,} GH failures)")

routing_df = pd.DataFrame(rows, columns=[
    "building_id", "park_id", "entrance_lat", "entrance_lon", "walk_dist_m"
])
routing_df["city"] = CITY

con.register("routing_tmp", routing_df)
con.execute(f"""
    CREATE OR REPLACE TABLE rule_300m.{CITY}_routing_raw AS
    SELECT * FROM routing_tmp
""")
con.unregister("routing_tmp")

con.execute(f"""
    CREATE OR REPLACE TABLE rule_300m.{CITY}_routing AS
    SELECT *
    FROM rule_300m.{CITY}_routing_raw
    QUALIFY ROW_NUMBER() OVER (PARTITION BY building_id ORDER BY walk_dist_m) = 1
""")

best = con.execute(f"SELECT * FROM rule_300m.{CITY}_routing").df()
print(f"{len(best):,} unique civici with a best route")
print(best["walk_dist_m"].describe().round(1).to_string())


Routing: 100%|██████████| 306436/306436 [1:06:52<00:00, 76.38it/s] 


306,436 pairs routed  (0 GH failures)
81,000 unique civici with a best route
count    81000.0
mean       156.9
std         87.7
min          0.0
25%         88.4
50%        154.8
75%        221.2
max       3516.6


## 10 final output

In [49]:
out_df = con.execute(f"""
    SELECT
        c.*,
        b.park_id       AS nearest_park_id,
        b.entrance_lat,
        b.entrance_lon,
        b.walk_dist_m,
        b.walk_dist_m IS NOT NULL              AS is_candidate,
        COALESCE(b.walk_dist_m <= {WALKING_DISTANCE}, FALSE) AS within_300m
    FROM raw.{CITY}_buildings_sn c
    LEFT JOIN rule_300m.{CITY}_routing b USING (building_id)
""").df()

out_df["geometry"] = out_df["geometry"].apply(bytes)
out = gpd.GeoDataFrame(
    out_df,
    geometry=gpd.array.from_wkb(out_df["geometry"]),
    crs=CRS_METRIC,
)

# Apply civici_in_park overrides
in_park_mask = out["building_id"].isin(civici_in_park_ids)
out.loc[in_park_mask, "within_300m"] = True
out.loc[in_park_mask, "walk_dist_m"] = out.loc[in_park_mask, "walk_dist_m"].fillna(0.0)
out.loc[in_park_mask, "nearest_park_id"] = (
    out.loc[in_park_mask, "building_id"]
    .map(civici_in_park_lookup)
    .combine_first(out.loc[in_park_mask, "nearest_park_id"])
)

# Rename
rename = {
    cfg["building_id_col"]: "street_number",
    "indirizzo_completo":   "address",
}
if cfg["neighbourhood_col"] and cfg["neighbourhood_col"] in out.columns:
    rename[cfg["neighbourhood_col"]] = "neighbourhood"
out = out.rename(columns=rename)

## 11 · Save  DuckDB + files

In [50]:
# DuckDB
gdf_to_duckdb(con, out, f"rule_300m.{CITY}_results")

# WGS84
out_wgs = out.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y

final_cols = [
    "street_number", "address", "longitude", "latitude",
    "neighbourhood", "nearest_park_id",
    "park_area_ha", "entrance_lat", "entrance_lon",
    "walk_dist_m", "within_300m", "geometry",
]
out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]

# GeoPackage
out.to_file(paths["gpkg"], driver="GPKG", layer="buildings")
parks[parks["park_id"].isin(out["nearest_park_id"].dropna())].to_file(
    paths["gpkg"], driver="GPKG", layer="parks"
)

# Exports
out_wgs.to_file(paths["geojson"], driver="GeoJSON")
out_wgs.drop(columns="geometry").to_csv(paths["csv"], index=False)

parquet_path = paths["csv"].replace(".csv", ".parquet")
out_wgs.drop(columns="geometry").to_parquet(parquet_path, index=False)

rule_300m.torino_results: 118824 rows written
